# 05 · Baselines B0-B5 (Fase 12)

Tabla comparativa de los baselines obligatorios más la representación ganadora de la Etapa B, bajo
el mismo protocolo (StratifiedGroupKFold de 5 folds agrupado por clúster de near-duplicados).

## 0. Configuración

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src import config as cfg, feature_catalog
from src.config import set_seed

set_seed()
pd.set_option("display.width", 175)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 60)

representaciones = json.loads((cfg.EXPERIMENTS / "representaciones_resultados.json").read_text())
baselines = json.loads((cfg.EXPERIMENTS / "baselines_resultados.json").read_text())
E001 = json.loads((cfg.EXPERIMENTS / "E-001_resultados.json").read_text())
print("resultados cargados:", len(representaciones), "representaciones |", len(baselines), "baselines")

resultados cargados: 14 representaciones | 3 baselines


## 1. Tabla de baselines

In [2]:
filas = [
    {"id": "B0", "descripcion": "DummyClassifier(prior)", "macro_f1": baselines["B0"]["macro_f1_mean"], "std": baselines["B0"]["macro_f1_std"]},
    {"id": "B1", "descripcion": "TF-IDF word 1-2 + LogisticRegression (OvR)", "macro_f1": baselines["B1"]["macro_f1_mean"], "std": baselines["B1"]["macro_f1_std"]},
    {"id": "B2", "descripcion": "TF-IDF word 1-2 + LinearSVC", "macro_f1": representaciones["E-002b"]["macro_f1_mean"], "std": representaciones["E-002b"]["macro_f1_std"]},
    {"id": "B3", "descripcion": "TF-IDF char_wb 3-5 + LinearSVC", "macro_f1": representaciones["E-003b"]["macro_f1_mean"], "std": representaciones["E-003b"]["macro_f1_std"]},
    {"id": "B4", "descripcion": "TF-IDF word 1-2 + char_wb 3-5 + LinearSVC", "macro_f1": E001["macro_f1_mean"], "std": E001["macro_f1_std"]},
    {"id": "B5", "descripcion": "35 features densas + LightGBM", "macro_f1": baselines["B5"]["macro_f1_mean"], "std": baselines["B5"]["macro_f1_std"]},
    {"id": "E-008", "descripcion": "char_wb 3-6 + canal de menciones + LinearSVC (ganadora)", "macro_f1": representaciones["E-008"]["macro_f1_mean"], "std": representaciones["E-008"]["macro_f1_std"]},
]
tabla = pd.DataFrame(filas)
tabla["macro_f1"] = tabla["macro_f1"].round(4)
tabla["std"] = tabla["std"].round(4)
tabla["vs_azar"] = (tabla["macro_f1"] - 1 / 6).round(4)
tabla.to_csv(cfg.TABLES / "baselines_comparativa.csv", index=False)
tabla

,id,descripcion,macro_f1,std,vs_azar
0,B0,DummyClassifier(prior),0.0485,0.0000,-0.1182
1,B1,TF-IDF word 1-2 + LogisticRegression (OvR),0.5015,0.0049,0.3348
2,B2,TF-IDF word 1-2 + LinearSVC,0.4971,0.0034,0.3304
3,B3,TF-IDF char_wb 3-5 + LinearSVC,0.5408,0.0050,0.3741
4,B4,TF-IDF word 1-2 + char_wb 3-5 + LinearSVC,0.5344,0.0050,0.3677
5,B5,35 features densas + LightGBM,0.3567,0.0036,0.1900
6,E-008,char_wb 3-6 + canal de menciones + LinearSVC (ganadora),0.5557,0.0052,0.3890


**OBSERVACIÓN.** El orden de mérito es claro: la señal dispersa de carácter domina (0,54-0,56), las
representaciones de palabra quedan en 0,50, las features densas en 0,36 y el azar en 0,17.

**INTERPRETACIÓN.** Las 35 features densas, pese a tener tamaños de efecto altos por separado
(ε² de `n_chars` = 0,126), no alcanzan a la representación dispersa porque comprimen el tweet en
pocos números. El baseline lineal disperso captura además el léxico y la ortografía.

**CONSECUENCIA.** La Fase 13 (modelos clásicos) debe trabajar sobre la representación dispersa
ganadora; las features densas se reservan para el análisis de errores y como complemento en un
posible ensamble (Fase 20).

## 2. Desempeño por clase de la representación ganadora

In [3]:
por_clase = pd.DataFrame({
    "B5_densas": baselines["B5"]["per_class_f1"],
    "B3_char_3_5": representaciones["E-003b"]["per_class_f1"],
    "E-008_ganadora": representaciones["E-008"]["per_class_f1"],
}).round(4)
por_clase["delta_ganadora_vs_B3"] = (por_clase["E-008_ganadora"] - por_clase["B3_char_3_5"]).round(4)
por_clase.to_csv(cfg.TABLES / "baselines_por_clase.csv")
por_clase

,B5_densas,B3_char_3_5,E-008_ganadora,delta_ganadora_vs_B3
13-17,0.4236,0.6141,0.6240,0.0099
18-24,0.3282,0.4436,0.4539,0.0103
25-34,0.2531,0.4292,0.4423,0.0131
35-49,0.2798,0.4957,0.5172,0.0215
50-64,0.3988,0.5958,0.6125,0.0167
65-xx,0.4570,0.6665,0.6845,0.0180


**LECTURA.** Las clases extremas (`13-17`, `65-xx`) son las mejor clasificadas —coherente con el
EDA: son las más distintas en registro y léxico— mientras que las clases contiguas
(`35-49`/`50-64`) concentran la confusión. Esto anticipa el análisis de errores (Fase 17) y el
ajuste de umbral por clase para macro-F1 (Fase 19).

**Cierre de la Etapa B.** Representación decidida por evidencia (D-010), baselines completos y
envío v02 preparado con la ganadora.